In [ ]:
import hashlib, importlib.metadata as md, json, os, shutil, subprocess, sys
from pathlib import Path
def atomic_json(path,payload):
    path.parent.mkdir(parents=True,exist_ok=True); tmp=path.with_name(path.name+".tmp")
    tmp.write_text(json.dumps(payload,sort_keys=True)+"\n",encoding="utf-8"); os.replace(tmp,path)
def parse_phase(raw):
    value=(raw or "").strip().lower()
    if value in {"","0","false","no"}: return "FAST_SAVE"
    if value in {"1","true","yes"}: return "FULL_RERUN"
    raise RuntimeError(f"unexpected rerun flag: {raw!r}")
def fast_save(challenge,submission):
    mounted=json.loads(challenge.read_text()); payload={task_id:[{"attempt_1":[[0]],"attempt_2":[[0]]} for _ in task["test"]] for task_id,task in mounted.items()}
    atomic_json(submission,payload)
    atomic_json(Path("/kaggle/working/artifacts/d1_failsoft/FAST_SAVE_PROVENANCE.json"),{"phase":"FAST_SAVE_ONLY","model_loaded":False,"submission_kind":"SAVE_ONLY_PLACEHOLDER","task_count":len(payload)})
    print(json.dumps({"event":"D1_FAILSOFT_FAST_SAVE_COMPLETE","task_count":len(payload)},sort_keys=True),flush=True)
def full_rerun(challenge,submission):
    submission.unlink(missing_ok=True)
    Path("/kaggle/working/artifacts/d1_failsoft/FAST_SAVE_PROVENANCE.json").unlink(missing_ok=True)
    mounted=Path("/kaggle/input/datasets/jimmy5566/arc2-d1-failsoft-v2-source")
    source_root=mounted/"ARC2"; manifest=json.loads((mounted/"SOURCE_MANIFEST.json").read_text())
    if manifest["archive_sha256"]!="50a3ad1ba6cb87e601bdf1f836621a23ac24a3415ca4bb6db652061e3081656b" or not source_root.is_dir(): raise RuntimeError("explicit fail-soft source identity mismatch")
    actual_files={Path(base,name).relative_to(source_root).as_posix() for base,_,files in os.walk(source_root) for name in files}
    if actual_files!=set(manifest["file_sha256"]): raise RuntimeError("fail-soft mounted source file set mismatch")
    for name,expected in manifest["file_sha256"].items():
        if hashlib.sha256((source_root/name).read_bytes()).hexdigest()!=expected: raise RuntimeError(f"fail-soft source hash mismatch: {name}")
    root=Path("/kaggle/working/ARC2")
    if root.exists(): shutil.rmtree(root)
    shutil.copytree(source_root,root)
    config=mounted/"d1_failsoft_release_config.json"
    if hashlib.sha256(config.read_bytes()).hexdigest()!=manifest["config_sha256"]: raise RuntimeError("fail-soft config hash mismatch")
    cfg=json.loads(config.read_text())
    os.environ.update({"TRITON_PTXAS_PATH":cfg["environment"]["ptxas_path"],"HF_HUB_OFFLINE":"1","TRANSFORMERS_OFFLINE":"1","TOKENIZERS_PARALLELISM":"false"})
    if not sys.version.startswith(cfg["environment"]["python_prefix"]): raise RuntimeError("Python mismatch")
    for package in ("unsloth","unsloth-zoo","transformers","torch","torchao","peft","trl","triton"):
        if md.version(package)!=cfg["environment"][package]: raise RuntimeError(f"dependency mismatch: {package}")
    ptxas=Path(cfg["environment"]["ptxas_path"]); model=Path("/kaggle/input/models/sorokin/qwen3_4b_grids15_sft139/transformers/bfloat16/1")
    if not ptxas.is_file() or subprocess.run([str(ptxas),"--version"],capture_output=True).returncode: raise RuntimeError("verified ptxas unavailable")
    out=Path("/kaggle/working/artifacts/d1_failsoft"); out.mkdir(parents=True,exist_ok=True)
    candidates=out/"candidates_frozen.json"; selection=out/"d1_selection_frozen.json"; provenance=out/"PRODUCTION_PROVENANCE.json"
    run=[sys.executable,str(root/"scripts/run_d1_failsoft_4gpu.py"),"--challenge",str(challenge),"--release-config",str(config),"--model-path",str(model),"--native-config-dir",str(root/"configs/nvarc_native_846d0198"),"--checkpoint-dir",str(out/"checkpoints"),"--output",str(candidates),"--resume"]
    if subprocess.run(run,env=os.environ).returncode: raise RuntimeError("D1_FAILSOFT_WORKERS_FAILED")
    final=[sys.executable,str(root/"scripts/build_d1_failsoft_submission.py"),"--challenge",str(challenge),"--release-config",str(config),"--records",str(candidates),"--selection-output",str(selection),"--provenance-output",str(provenance),"--output",str(submission)]
    if subprocess.run(final,env={**os.environ,"CUDA_VISIBLE_DEVICES":""}).returncode: raise RuntimeError("D1_FAILSOFT_FINALIZATION_FAILED")
    payload=json.loads(submission.read_text()); mounted_challenge=json.loads(challenge.read_text())
    if set(payload)!=set(mounted_challenge) or any(len(payload[k])!=len(mounted_challenge[k]["test"]) for k in mounted_challenge): raise RuntimeError("final submission mapping mismatch")
    print(json.dumps({"event":"D1_FAILSOFT_RELEASE_COMPLETE","task_count":len(payload),"solutions_opened":False},sort_keys=True),flush=True)
def main():
    challenge=Path("/kaggle/input/competitions/arc-prize-2026-arc-agi-2/arc-agi_test_challenges.json")
    if not challenge.is_file(): raise RuntimeError("mounted competition challenge missing")
    submission=Path("/kaggle/working/submission.json")
    phase=parse_phase(os.getenv("KAGGLE_IS_COMPETITION_RERUN", ""))
    fast_save(challenge,submission) if phase=="FAST_SAVE" else full_rerun(challenge,submission)
main()
